# Session 1: thrust, weight and why a drone cannot hover on a fixed throttle
**A. Real-world problem.** A delivery drone of 1.2 kg must hover still above a doorstep.
Its four rotors push air down and the air pushes the drone up. How fast must the rotors
spin to hover? And if someone clips on a 0.2 kg parcel and the pilot does not touch the
throttle, what happens next?

**Plan.** (1) weight and rotor thrust, (2) the hover speed of each rotor, (3) thrust-to-weight
ratio, (4) Newton's second law in the vertical direction, (5) what a fixed throttle does when
the mass is not what we assumed. This last step is why every drone needs feedback.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

G = 9.81          # gravitational acceleration (m/s^2)
K_T = 1.0e-5      # rotor thrust coefficient (N per (rad/s)^2), typical of a 10-inch propeller
RHO = 1.225       # density of air at sea level (kg/m^3)

## Step 1: weight and rotor thrust
Gravity pulls the drone down with its weight W = m g (newtons, N). A spinning propeller
throws air downwards; by Newton's third law the air pushes the propeller up. For a fixed
propeller this thrust grows with the **square** of the spin speed omega (rad/s):

    T = k_T omega^2

Double the speed and you get four times the thrust. k_T depends on the propeller's size,
shape and the air density; we use 1.0e-5 N/(rad/s)^2.

In [ ]:
def weight(mass, g=G):
    """Weight in N of a mass in kg."""
    if mass <= 0:
        raise ValueError("mass must be positive")
    return mass * g


def rotor_thrust(omega, k_t=K_T):
    """Thrust (N) of one rotor spinning at omega rad/s: T = k_T omega^2."""
    return k_t * np.asarray(omega, float) ** 2


def rpm_from_omega(omega):
    """Convert rad/s to revolutions per minute."""
    return np.asarray(omega, float) * 60.0 / (2.0 * np.pi)

## Step 2: the hover speed
To hover, the total thrust of the n rotors must equal the weight, so each rotor carries
W / n and spins at

    omega_hover = sqrt(m g / (n k_T))

In [ ]:
def hover_omega(mass, n_rotors=4, k_t=K_T, g=G):
    """Spin speed (rad/s) at which n equal rotors exactly carry the weight."""
    if n_rotors < 1:
        raise ValueError("need at least one rotor")
    return np.sqrt(weight(mass, g) / (n_rotors * k_t))

## Step 3: thrust-to-weight ratio
A drone that can only just carry itself cannot climb, fight a gust or recover from a tilt.
The thrust-to-weight ratio is the most total thrust divided by the weight. Builders aim for
at least 2; racing drones go above 5.

In [ ]:
def thrust_to_weight(mass, omega_max, n_rotors=4, k_t=K_T, g=G):
    """Largest total thrust divided by weight (no unit)."""
    return n_rotors * float(rotor_thrust(omega_max, k_t)) / weight(mass, g)

## Step 4: Newton's second law, up and down
Take "up" as positive. The net force is total thrust minus weight, so

    m a = T_total - m g        a = T_total / m - g

If the thrust is held fixed, a is constant and, starting at rest, the height changes by
z(t) = a t^2 / 2. A small mistake in thrust grows into a large mistake in height, because
the error is squared in time.

In [ ]:
def vertical_acceleration(total_thrust, mass, g=G):
    """Vertical acceleration (m/s^2, up positive) for a given total thrust."""
    return total_thrust / mass - g


def open_loop_height(t, total_thrust, mass, g=G):
    """Height change (m) after time t (s), from rest, with the thrust held fixed."""
    a = vertical_acceleration(total_thrust, mass, g)
    return 0.5 * a * np.asarray(t, float) ** 2


def time_to_drop(distance, total_thrust, mass, g=G):
    """Time (s) to sink a distance (m) from rest. Returns inf if the drone does not sink."""
    a = vertical_acceleration(total_thrust, mass, g)
    if a >= 0:
        return float("inf")
    return float(np.sqrt(2.0 * distance / -a))

## Step 5 (extra): hovering costs power even when nothing moves
Momentum theory says a rotor of disc area A that makes thrust T in still air needs at least
the ideal induced power P = T^1.5 / sqrt(2 rho A). Real rotors need roughly 1.5 times more.
A heavier drone needs more thrust and therefore more power: (1.4 / 1.2)^1.5 = 1.26, so a
0.2 kg parcel costs about 26 percent more hover power.

In [ ]:
def ideal_hover_power(thrust, radius, rho=RHO):
    """Ideal induced power (W) of one rotor of given radius (m) making a thrust (N)."""
    area = np.pi * radius ** 2
    return float(thrust) ** 1.5 / np.sqrt(2.0 * rho * area)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m, m_parcel = 1.2, 0.2
    w_h = hover_omega(m)
    print(f"weight of a {m} kg drone: {weight(m):.3f} N, per rotor {weight(m) / 4:.3f} N")
    print(f"hover speed: {w_h:.1f} rad/s = {float(rpm_from_omega(w_h)):.0f} rpm")
    print(f"thrust-to-weight at 1000 rad/s: {thrust_to_weight(m, 1000.0):.2f}")
    print(f"ideal hover power, 0.127 m rotors: {4 * ideal_hover_power(weight(m) / 4, 0.127):.1f} W")

    # Figure 1: thrust against spin speed, and the hover point for two masses
    omega = np.linspace(0, 1000, 300)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(rpm_from_omega(omega), rotor_thrust(omega), color="#1f6feb", lw=2.5, label="one rotor, T = k_T omega^2")
    for mass, col in [(m, "#2a9d8f"), (m + m_parcel, "#d9534f")]:
        t_each = weight(mass) / 4
        rpm = float(rpm_from_omega(hover_omega(mass)))
        ax.axhline(t_each, color=col, ls="--", lw=1)
        ax.plot(rpm, t_each, "o", color=col, label=f"{mass:.1f} kg hover: {t_each:.2f} N at {rpm:.0f} rpm")
    ax.set_xlabel("rotor speed (rpm)")
    ax.set_ylabel("thrust of one rotor (N)")
    ax.set_title("Thrust grows with the square of the rotor speed")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(OUT / "thrust_curve.png", dpi=120)

    # Figure 2: a fixed throttle set for 1.2 kg, flown with three different real masses
    t = np.linspace(0, 2.0, 200)
    thrust = weight(m)          # the pilot set exactly the hover thrust of a 1.2 kg drone
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for mass, col, lab in [(m - m_parcel, "#2a9d8f", "1.0 kg (lighter)"), (m, "k", "1.2 kg (as assumed)"),
                           (m + m_parcel, "#d9534f", "1.4 kg (parcel added)")]:
        z = open_loop_height(t, thrust, mass)
        ax2.plot(t, z, color=col, lw=2, label=f"{lab}: a = {vertical_acceleration(thrust, mass):+.2f} m/s^2")
        print(f"{lab}: height after 2 s = {z[-1]:+.2f} m")
    ax2.axhline(-1.0, color="#999999", ls=":", lw=1)
    ax2.text(0.05, -0.92, "1 m below start", fontsize=8, color="#666666")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("height change z (m)")
    ax2.set_title("Fixed throttle: a wrong mass means a growing height error")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8, loc="lower left")
    fig2.tight_layout()
    fig2.savefig(OUT / "open_loop_drift.png", dpi=120)
    print(f"with the parcel the drone sinks 1 m in {time_to_drop(1.0, thrust, m + m_parcel):.2f} s")

## Where this stops
A fixed throttle only works if we know the mass, the air density and the battery voltage
exactly, and we never do. Session 2 measures the height and corrects the thrust all the
time: feedback, with a PID controller.

## Coding tasks
1. **Easy.** In `lesson.py` change the parcel `m_parcel` to 0.1, 0.3 and 0.5 kg. Predict the hover rpm of each from the square-root law, then run it and compare.
2. **Stretch.** Write `hexa_hover_omega(mass)` for a six-rotor drone and check that it spins sqrt(4 / 6) = 0.816 times as fast as the quadrotor at the same mass.
3. **Explain.** Explain why a 17 percent heavier drone needs only about 8 percent more rotor speed, but about 26 percent more hover power.

## Assessment
- **A. Conceptual.** A pilot finds the exact throttle at which the drone hovers, then lets go of the stick. Give three reasons why the drone will still drift up or down after a while.
- **B. Numerical.** A smaller 0.9 kg quadrotor uses the same rotors (k_T = 1.0e-5 N/(rad/s)^2, top speed 1000 rad/s). Find the thrust per rotor to hover, the hover speed in rad/s and rpm, and its thrust-to-weight ratio. Check the units and that the answer is reasonable.
- **C. Graph and simulation interpretation.** On the thrust curve, by what factor does the thrust grow when the rotor speed goes from 4000 to 8000 rpm? On the drift graph, the 1.4 kg drone has sunk 2.80 m after 2 s. How far had it sunk after 1 s, and why is that not half?
- **D. Design and optimisation challenge.** Design: you want the 1.2 kg drone to have a thrust-to-weight ratio of at least 2. What top rotor speed (rpm) is needed with k_T = 1.0e-5? What top speed is needed if it must also carry a 0.4 kg parcel? With the real top speed of 1000 rad/s, what is the largest payload that keeps the ratio at 2?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: a 1.2 kg drone weighs 11.77 N, 2.943 N per rotor",
                   abs(L.weight(1.2) - 11.772) < 1e-9 and abs(L.weight(1.2) / 4 - 2.943) < 1e-9))
    w_h = L.hover_omega(1.2)
    checks.append(("worked example: hover speed 542.5 rad/s (about 5180 rpm)",
                   abs(w_h - 542.49) < 0.05 and abs(float(L.rpm_from_omega(w_h)) - 5180) < 1.0))
    checks.append(("at the hover speed the four rotors carry exactly the weight",
                   abs(4 * float(L.rotor_thrust(w_h)) - L.weight(1.2)) < 1e-9))
    checks.append(("thrust grows with the square of speed: doubling omega gives 4 times the thrust",
                   abs(float(L.rotor_thrust(800.0)) / float(L.rotor_thrust(400.0)) - 4.0) < 1e-12))
    checks.append(("hover speed scales with the square root of mass: 4x the mass needs 2x the speed",
                   abs(L.hover_omega(4.8) / L.hover_omega(1.2) - 2.0) < 1e-12))
    checks.append(("hand calculation: thrust-to-weight at 1000 rad/s is 40 / 11.772 = 3.40",
                   abs(L.thrust_to_weight(1.2, 1000.0) - 40.0 / 11.772) < 1e-9))
    checks.append(("exact hover thrust gives zero acceleration", abs(L.vertical_acceleration(L.weight(1.2), 1.2)) < 1e-12))
    checks.append(("zero thrust gives free fall, a = -g", abs(L.vertical_acceleration(0.0, 1.4) + 9.81) < 1e-12))
    a = L.vertical_acceleration(L.weight(1.2), 1.4)
    checks.append(("worked example: with a 0.2 kg parcel a = -1.40 m/s^2", abs(a + 1.4014) < 1e-3))
    checks.append(("worked example: it sinks 1 m in 1.19 s", abs(L.time_to_drop(1.0, L.weight(1.2), 1.4) - 1.195) < 2e-3))
    checks.append(("height from z = a t^2 / 2 agrees with time_to_drop",
                   abs(L.open_loop_height(L.time_to_drop(1.0, L.weight(1.2), 1.4), L.weight(1.2), 1.4) + 1.0) < 1e-9))
    checks.append(("a drone that does not sink never drops", L.time_to_drop(1.0, L.weight(1.2), 1.0) == float("inf")))
    p12 = L.ideal_hover_power(L.weight(1.2) / 4, 0.127)
    p14 = L.ideal_hover_power(L.weight(1.4) / 4, 0.127)
    checks.append(("hover power grows as mass^1.5: the parcel costs 26 percent more", abs(p14 / p12 - (1.4 / 1.2) ** 1.5) < 1e-12
                   and 1.25 < p14 / p12 < 1.27))
    checks.append(("hand calculation: ideal hover power of the 1.2 kg drone is about 57 W",
                   abs(4 * p12 - 57.3) < 0.2))
    bad = 0
    for call in (lambda: L.weight(0.0), lambda: L.hover_omega(1.2, n_rotors=0), lambda: L.weight(-1.0)):
        try:
            call()
        except ValueError:
            bad += 1
    checks.append(("invalid inputs raise ValueError", bad == 3))
    _ = np  # numpy is available for your own extra checks
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')